# Trapping and pitch modes

Three pieces behind the current drive: the G-EQDSK files that carry an equilibrium in and out, the orbit-averaged pitch of the fast ions with its eigenmodes of pitch-angle scattering (`freeofcharge.trapping`), and the trapped-electron correction of the shielding current (Sauter or Start-Cordey).

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import plasma_plots
import xarray as xr

import freeofcharge as foc
from freeofcharge import geqdsk
from freeofcharge.slowing_down import (
    l31_coefficient,
    sauter_trapped_electron_correction,
    start_cordey_fit,
    start_cordey_trapped_electron_correction,
)
from freeofcharge.trapping import pitch_modes, transit_pitch

equilibrium = foc.Equilibrium.solovev(6.2, 1.8, 1.7, 5.3, wall_psi_n=1.3)
surfaces = equilibrium.flux_surfaces(np.linspace(0.0, 1.0, 21))

## G-EQDSK files

`Equilibrium.to_geqdsk` and `Equilibrium.from_geqdsk` cover the usual round trip. Below them, `freeofcharge.geqdsk.read` returns a `GEqdsk` with every part of the file (the grid, `psirz`, `fpol`, the axis, the boundary and the limiter, and the pressure and `q` that `Equilibrium` does not use), and `geqdsk.write` writes one back. Edit a file this way: here the safety factor of the flux surfaces is added to a file that `to_geqdsk` wrote with `q` left zero.

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    path = Path(directory) / "solovev.geqdsk"
    equilibrium.to_geqdsk(path, description="Solov'ev ITER-like")
    print(path.read_text().splitlines()[0])

    data = geqdsk.read(path)
    print(
        f"grid {data.psi.shape}, axis {data.axis}, boundary {len(data.boundary)} points"
    )
    psi_n = np.linspace(0.0, 1.0, len(data.fpol))
    rho_q = np.sqrt(psi_n)
    data.q = np.interp(rho_q, surfaces.rho, surfaces.safety_factor)
    data.description = "with q"
    geqdsk.write(path, data)

    edited = geqdsk.read(path)
    reread = foc.Equilibrium.from_geqdsk(path)

assert edited.description.strip() == "with q"
assert np.allclose(edited.q, data.q, rtol=1e-8)
assert np.allclose(edited.psi, data.psi, rtol=1e-8, atol=1e-12)
r, z = 7.0, 0.5
assert abs(float(reread.psi(r, z)) - float(equilibrium.psi(r, z))) < 1e-6 * abs(
    equilibrium.psi_boundary
)

The numbers in a G-EQDSK file may run together (`1.0e+00-2.0e+00`), so `read` takes them by pattern rather than splitting on spaces; files from EFIT and CHEASE read as they are.

## Trapped particles

On each flux surface, `trapped_fraction` integrates the field strength along the surface (Lin-Liu and Miller 1995); at large aspect ratio it approaches `1.46 sqrt(eps)`. A fast ion with `lambda = (1 - xi^2) B_min / B` above `B_min / B_max` is trapped; a passing one carries the transit-averaged pitch `<xi>_t` along the field (`transit_pitch`), smaller than its local pitch, and zero once trapped.

In [ ]:
eps = surfaces.inverse_aspect_ratio
fraction = xr.Dataset(
    {
        "trapped_fraction": ("rho", surfaces.trapped_fraction),
        "large_aspect_ratio": ("rho", 1.46 * np.sqrt(eps)),
    },
    coords={"rho": surfaces.rho},
).to_array("model", name="f_t")

shell = 10
strength = surfaces.field_strength[shell]
b = strength / strength.min()
line_weight = surfaces.surface_weight[shell] * b
line_weight /= line_weight.sum()
xi = np.linspace(0.0, 1.0, 201)
transit = xr.DataArray(
    transit_pitch(b, line_weight, 1.0 - xi**2),
    dims="xi",
    coords={"xi": xi},
    name="<xi>_t",
    attrs={"long_name": "transit-averaged pitch"},
).expand_dims(rho=[round(float(surfaces.rho[shell]), 3)])
xi_c = np.sqrt(1.0 - 1.0 / b.max())
print(f"trapped below xi = {xi_c:.3f} at the outboard midplane of shell {shell}")
assert np.all(transit.values[0, xi < xi_c - 1e-3] == 0.0)
assert np.all(transit.values[0] <= xi + 1e-12)

with plasma_plots.figure(1, 2, figsize=(10, 3.5)) as fig:
    fraction.plasma.plot.profiles(x="rho", over="model", ax=fig[0])
    transit.plasma.plot.profiles(x="xi", over="rho", ax=fig[1])

## Pitch modes

Pitch-angle scattering averaged over the orbits keeps the trapped ions at zero mean pitch and relaxes the passing ones through odd eigenmodes `F_k(lambda)`, each decaying as `exp(-kappa_k int nu_D dt)`. `pitch_modes(surfaces)` solves the eigenproblem on every shell. In a uniform field the modes are the odd Legendre polynomials with `kappa = l (l + 1) / 2 = 1, 6, 15, ...`; trapping makes them decay faster, the more so the further out. `slow_down` uses these modes for the current of the orbits (with `orbit_average=True`, the default) instead of Gaffey's single exponential.

In [ ]:
modes = pitch_modes(surfaces, modes=4)
rates = xr.DataArray(
    modes.rates[:, :4],
    dims=("rho", "mode"),
    coords={"rho": surfaces.rho, "mode": np.arange(1, 5)},
    name="kappa",
    attrs={"long_name": "decay rate / nu_D"},
)
print("innermost shell:", np.round(modes.rates[0, :3], 2), "uniform field: 1, 6, 15")
assert np.all(np.diff(modes.rates[:, 0]) > 0)  # more trapping, faster decay

shapes = xr.DataArray(
    modes.modes[shell, :3],
    dims=("mode", "lambda"),
    coords={"mode": [1, 2, 3], "lambda": modes.lam[shell]},
    name="F_k",
    attrs={"long_name": f"pitch mode at rho = {surfaces.rho[shell]:.3f}"},
)
with plasma_plots.figure(1, 2, figsize=(10, 3.5)) as fig:
    rates.plasma.plot.profiles(x="rho", over="mode", ax=fig[0])
    shapes.plasma.plot.profiles(x="lambda", over="mode", ax=fig[1])

`modes.amplitudes(shell, pitch, field_strength)` splits the transit-averaged pitch of each birth into the modes; their sum is `<xi>_t`, and the modes beyond those kept go into a remainder.

In [ ]:
pitch = np.array([0.9, 0.6, -0.6, 0.2])
field = np.full(pitch.shape, strength.min())
amplitudes = modes.amplitudes(np.full(pitch.shape, shell), pitch, field)
expected = np.sign(pitch) * transit_pitch(b, line_weight, 1.0 - pitch**2)
print(np.round(amplitudes, 3))
assert np.allclose(amplitudes.sum(axis=1), expected)

## Trapped-electron correction

The fast ions drag the electrons along, whose return current cancels most of theirs; trapped electrons cannot follow, so the net current is `1 - Z_b / Z_eff (1 - G)` times that of the fast ions. `slow_down` takes `G` from `trapped_electron_correction`: by default `sauter_trapped_electron_correction`, Sauter's `L31` with the trapped fraction and the electron collisionality of each shell, or `start_cordey_trapped_electron_correction`, the collisionless large-aspect-ratio fit to Start and Cordey. Without collisions the two agree within a few percent up to `eps = 0.3`; collisions lower Sauter's `G` at the edge, where the plasma is cold and dense.

In [ ]:
plasma = foc.Plasma.parabolic(1e20, 2e4, ions={"D": (2, 1, 0.96), "He": (4, 2, 0.02)})
zeff = np.asarray(plasma.zeff(surfaces.rho))
f_t = 1.46 * np.sqrt(eps) - 0.46 * eps**1.5
collisionless = np.asarray(l31_coefficient(zeff, f_t))
fit = np.asarray(start_cordey_fit(zeff, eps))
small = eps < 0.3
assert np.all(np.abs(fit[small] / collisionless[small] - 1) < 0.06)

correction = xr.Dataset(
    {
        "Sauter": ("rho", sauter_trapped_electron_correction(surfaces, plasma)),
        "Sauter, no collisions": (
            "rho",
            np.asarray(l31_coefficient(zeff, surfaces.trapped_fraction)),
        ),
        "Start-Cordey": (
            "rho",
            start_cordey_trapped_electron_correction(surfaces, plasma),
        ),
    },
    coords={"rho": surfaces.rho},
).to_array("model", name="G")
with plasma_plots.figure(figsize=(5.5, 3.5)) as fig:
    correction.plasma.plot.profiles(x="rho", over="model", ax=fig[0])

The choice changes the driven current. A small co-current beam:

In [ ]:
origin, direction = foc.tangential_injection(5.3, 12.0)
beam = foc.NeutralBeam.gaussian(
    origin, direction, 1.0e6, 16.5e6, width=0.3, divergence=0.005, markers=300, seed=1
)
deposition = foc.deposit(
    beam,
    equilibrium,
    plasma,
    step=0.03,
    births_per_marker=2,
    rho_edges=np.linspace(0.0, 1.0, 21),
    seed=2,
)
currents = {
    name: foc.slow_down(
        deposition, plasma, surfaces, trapped_electron_correction=correction_model
    ).driven_current
    for name, correction_model in [
        ("Sauter", sauter_trapped_electron_correction),
        ("Start-Cordey", start_cordey_trapped_electron_correction),
    ]
}
for name, current in currents.items():
    print(f"{name:13s} {current / 1e3:7.1f} kA")